# TF-IDF + Logistic Regression + sublinear_tf=True

Experimento de classificação da clareza das respostas utilizando TF-IDF com ponderação sublinear da frequência dos termos e Regressão Logística.

In [3]:
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

In [4]:
!git clone https://github.com/juliaduboisas/plnEp1.git
%cd plnEp1
!ls

Cloning into 'plnEp1'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 88 (delta 30), reused 63 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 14.54 MiB | 14.16 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/content/plnEp1
baselineModel.ipynb	     tf_idf_reglog.ipynb
bertimbauModel.ipynb	     tfidfregord.ipynb
dataExploration.ipynb	     tfidf_sublinear.ipynb
ep1-enunciado.pdf	     TF_IDF_Word_+_Linear_SVM.ipynb
exponentialregression.ipynb  TF_IDF_+_XGBoost.ipynb
laya_custom		     train.csv
layaFineTune.ipynb	     train_noDuplicates.csv
pipelines		     train_noSameClassDuplicates.csv
README.md		     useModel.ipynb
roBERTaModel.ipynb	     word2vecmlp.ipynb
sbertModel.ipynb	     word2vecreglog.ipynb


## 1. Carregamento dos dados

In [5]:
df = pd.read_csv("train.csv")

Y = df["clarity"]

print(df.shape)
df.head()

(20092, 2)


,resp_text,clarity
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c5
1,"Prezada cidadã, As informações sobre óbitos ...",c1
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",c1
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c234
4,"Senhor, O Serviço de Informações ao Cidadão d...",c234


## 2. Vetorização TF-IDF

Neste experimento, utilizamos `sublinear_tf=True`, que aplica uma ponderação sublinear à frequência dos termos.

In [6]:
vect = TfidfVectorizer(
    analyzer="word",
    sublinear_tf=True,
    ngram_range=(1, 3)
)

X = vect.fit_transform(df["resp_text"])

print(X.shape)

(20092, 1473992)


## 3. Separação dos dados

In [7]:
x_train, x_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=123,
    stratify=Y
)

print("Treino:", x_train.shape)
print("Teste:", x_test.shape)

Treino: (16073, 1473992)
Teste: (4019, 1473992)


## 4. Regressão Logística

In [8]:
clf = LogisticRegression(
    C=1,
    class_weight="balanced"
)

clf.fit(
    x_train,
    y_train
)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(C=1, class_weight='balanced')

## 5. Avaliação

In [9]:
predicted = clf.predict(x_test)

score = f1_score(
    y_test,
    predicted,
    average="macro"
)

accuracy = accuracy_score(
    y_test,
    predicted
)

print(f"Macro F1: {score:.4f}")
print(f"Accuracy: {accuracy:.4f}")

Macro F1: 0.4771
Accuracy: 0.4800


In [10]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        predicted
    )
)

              precision    recall  f1-score   support

          c1       0.51      0.51      0.51      1269
        c234       0.41      0.36      0.38      1371
          c5       0.51      0.58      0.54      1379

    accuracy                           0.48      4019
   macro avg       0.48      0.48      0.48      4019
weighted avg       0.48      0.48      0.48      4019



In [11]:
# ============================================================
# VALIDAÇÃO CRUZADA — TF-IDF + Logistic Regression
# Melhor configuração encontrada
# ============================================================

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate


# ------------------------------------------------------------
# 1. Dados
# ------------------------------------------------------------

df = pd.read_csv("train.csv")

X = df["resp_text"]
y = df["clarity"]


# ------------------------------------------------------------
# 2. Melhor configuração encontrada
# ------------------------------------------------------------

pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="word",
            sublinear_tf=True,
            ngram_range=(1, 3)
        )
    ),
    (
        "logreg",
        LogisticRegression(
            C=1.0,
            class_weight="balanced"
        )
    )
])


# ------------------------------------------------------------
# 3. Cross-validation estratificada
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)


# ------------------------------------------------------------
# 4. Avaliação
# ------------------------------------------------------------

results = cross_validate(
    pipeline,
    X,
    y,
    cv=cv,
    scoring={
        "f1_macro": "f1_macro",
        "accuracy": "accuracy"
    },
    return_train_score=True,
    n_jobs=-1
)


# ------------------------------------------------------------
# 5. Resultados por fold
# ------------------------------------------------------------

print("Resultados por fold:\n")

for i in range(5):
    print(
        f"Fold {i+1}: "
        f"F1 Macro = {results['test_f1_macro'][i]:.4f} | "
        f"Accuracy = {results['test_accuracy'][i]:.4f}"
    )


# ------------------------------------------------------------
# 6. Média e desvio padrão
# ------------------------------------------------------------

print("\nResultado da validação cruzada:")

print(
    f"Macro F1: "
    f"{results['test_f1_macro'].mean():.4f} "
    f"± {results['test_f1_macro'].std():.4f}"
)

print(
    f"Accuracy: "
    f"{results['test_accuracy'].mean():.4f} "
    f"± {results['test_accuracy'].std():.4f}"
)

print("\nTreino:")
print(
    f"Macro F1: "
    f"{results['train_f1_macro'].mean():.4f} "
    f"± {results['train_f1_macro'].std():.4f}"
)

print(
    f"Accuracy: "
    f"{results['train_accuracy'].mean():.4f} "
    f"± {results['train_accuracy'].std():.4f}"
)

Resultados por fold:

Fold 1: F1 Macro = 0.4584 | Accuracy = 0.4598
Fold 2: F1 Macro = 0.4669 | Accuracy = 0.4680
Fold 3: F1 Macro = 0.4650 | Accuracy = 0.4652
Fold 4: F1 Macro = 0.4562 | Accuracy = 0.4567
Fold 5: F1 Macro = 0.4662 | Accuracy = 0.4664

Resultado da validação cruzada:
Macro F1: 0.4625 ± 0.0044
Accuracy: 0.4632 ± 0.0043

Treino:
Macro F1: 0.8574 ± 0.0014
Accuracy: 0.8575 ± 0.0014
